# Tarea 13 - Agente RAG

### Tema principal: Energías renovables
### Tema oculto: Aeronaves comerciales

___

___

## Dependencias

In [61]:
from pathlib import Path

import ollama
import chromadb
import pymupdf
from ddgs import DDGS

from sentence_transformers import SentenceTransformer

___

___

## Configuración

In [62]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Paths
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ROOT = Path.cwd()
DOCS_DIR = ROOT / "docs"
CHROMA_DIR = ROOT / "vectors"
COLLECTION_NAME = "nlp_docs"

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Modelos
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OLLAMA_MODEL = "granite4.1:3b"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Chunking
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CHUNK_SIZE = 700
CHUNK_OVERLAP = 150

___

___

## Sistema RAG

### Documentos

In [63]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Carga de documentos
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def load_pdf_documents(docs_dir: Path) -> list[dict]:
    documents = []

    pdf_files = sorted(docs_dir.glob("*.pdf"))

    for pdf_path in pdf_files:
        pdf_document = pymupdf.open(pdf_path)

        for page_index in range(len(pdf_document)):
            page = pdf_document[page_index]

            text = page.get_text().strip()

            if text:
                documents.append(
                    {
                        "source": pdf_path.name,
                        "page": page_index + 1,
                        "text": text
                    }
                )

        pdf_document.close()

    return documents

documents = load_pdf_documents(DOCS_DIR)

In [64]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Chunking de documentos
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def split_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

def create_pdf_chunks(documents: list[dict]) -> list[dict]:
    chunks = []

    for document in documents:
        text_chunks = split_text(
            text=document["text"],
            chunk_size=CHUNK_SIZE,
            overlap=CHUNK_OVERLAP
        )

        for chunk_index, chunk_text in enumerate(text_chunks):
            chunk_id = f"{document['source']}_page_{document['page']}_chunk_{chunk_index}"

            chunks.append(
                {
                    "id": chunk_id,
                    "source": document["source"],
                    "page": document["page"],
                    "chunk_index": chunk_index,
                    "text": chunk_text
                }
            )

    return chunks

chunks = create_pdf_chunks(documents)

print("Chunks creados:", len(chunks))

for chunk in chunks[:3]:
    print("ID:", chunk["id"])
    print("SOURCE:", chunk["source"])
    print("PAGE:", chunk["page"])
    print("TEXT:", chunk["text"][:500])
    print("-" * 80)

Chunks creados: 2578
ID: 27955.pdf_page_1_chunk_0
SOURCE: 27955.pdf
PAGE: 1
TEXT: What is Renewable Energy?
Renewable energy uses energy sources
that are continually replenished by
nature—the sun, the wind, water, the
Earth’s heat, and plants. Renewable
energy technologies turn these fuels into
usable forms of energy—most often elec-
tricity, but also heat, chemicals, or
mechanical power.
Why Use Renewable Energy?
Today we primarily use fossil fuels to heat
and power our homes and fuel our cars.
It’s convenient to use coal, oil, and natural
gas for meeting our energy needs, b
--------------------------------------------------------------------------------
ID: 27955.pdf_page_1_chunk_1
SOURCE: 27955.pdf
PAGE: 1
TEXT: Earth. We’re using them much more
rapidly than they are being created. Even-
tually, they will run out. And because of 
safety concerns and waste disposal prob-
lems, the United States will retire much of
its nuclear capacity by 2020. In the mean-
time, the nation’s energy n

### ChromaDB

In [65]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Cliente de ChromaDB
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_DIR)
)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Colección de documentos
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME
)



# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Indexación de documentos
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def index_pdf_chunks(collection, chunks: list[dict], embedding_model) -> None:
    ids = []
    documents = []
    metadatas = []
    embeddings = []

    for chunk in chunks:
        ids.append(chunk["id"])
        documents.append(chunk["text"])

        metadatas.append(
            {
                "source": chunk["source"],
                "page": chunk["page"],
                "chunk_index": chunk["chunk_index"]
            }
        )

        embedding = embedding_model.encode(chunk["text"]).tolist()

        embeddings.append(embedding)

    if ids:
        collection.upsert(
            ids=ids,
            documents=documents,
            metadatas=metadatas,
            embeddings=embeddings
        )


embedding_model = SentenceTransformer(EMBEDDING_MODEL)
index_pdf_chunks(
    collection=collection,
    chunks=chunks,
    embedding_model=embedding_model
)

print("Chunks indexados en Chroma:", collection.count())

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 16098.28it/s]


Chunks indexados en Chroma: 2578


### RAG Tool

In [66]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Búsqueda de documentos relevantes
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def search_pdf_rag(query: str, top_k: int = 3) -> list[dict]:
    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    retrieved_chunks = []

    if not results["ids"] or not results["ids"][0]:
        return retrieved_chunks

    for index in range(len(results["ids"][0])):
        retrieved_chunks.append(
            {
                "id": results["ids"][0][index],
                "text": results["documents"][0][index],
                "metadata": results["metadatas"][0][index],
                "distance": results["distances"][0][index]
            }
        )

    return retrieved_chunks


def build_pdf_context(retrieved_chunks: list[dict]) -> str:
    context_parts = []

    for chunk in retrieved_chunks:
        source = chunk["metadata"]["source"]
        page = chunk["metadata"]["page"]
        text = chunk["text"]

        context_parts.append(
            f"Fuente: {source}, página {page}\nContenido:\n{text}"
        )

    return "\n\n".join(context_parts)

In [67]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Definición de la tool
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
RAG_TOOL_DEFINITION = {
    "name": "rag_search_tool",
    "description": (
        "Busca información relevante en los documentos PDF locales "
        "usando Chroma y embeddings con all-MiniLM-L6-v2."
    ),
    "input": {
        "query": "Pregunta o consulta del usuario en lenguaje natural.",
        "top_k": "Número de fragmentos relevantes a recuperar."
    },
    "output": {
        "tool_name": "Nombre de la herramienta ejecutada.",
        "found": "Indica si se encontró información potencialmente útil.",
        "context": "Contexto construido con los chunks recuperados.",
        "sources": "Fuentes PDF, páginas, chunk index y distancia.",
        "best_distance": "Distancia del resultado más relevante."
    }
}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# RAG tool
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def rag_search_tool(query: str, top_k: int = 3) -> dict:
    retrieved_chunks = search_pdf_rag(
        query=query,
        top_k=top_k
    )

    if not retrieved_chunks:
        return {
            "tool_name": "rag_search_tool",
            "found": False,
            "context": "",
            "sources": [],
            "best_distance": None
        }

    context = build_pdf_context(retrieved_chunks)

    sources = []

    for chunk in retrieved_chunks:
        sources.append(
            {
                "source": chunk["metadata"]["source"],
                "page": chunk["metadata"]["page"],
                "chunk_index": chunk["metadata"]["chunk_index"],
                "distance": chunk["distance"]
            }
        )

    best_distance = min(
        chunk["distance"] for chunk in retrieved_chunks
    )

    return {
        "tool_name": "rag_search_tool",
        "found": True,
        "context": context,
        "sources": sources,
        "best_distance": best_distance
    }

___

___

## Web Search

In [71]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Definición de la tool
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
WEB_TOOL_DEFINITION = {
    "name": "web_search_tool",
    "description": (
        "Busca información en internet usando DuckDuckGo mediante ddgs. "
        "Debe utilizarse cuando el sistema RAG local no contiene información suficiente "
        "para responder la pregunta del usuario."
    ),
    "input": {
        "query": "Pregunta o consulta del usuario en lenguaje natural.",
        "max_results": "Número máximo de resultados de búsqueda a recuperar."
    },
    "output": {
        "tool_name": "Nombre de la herramienta ejecutada.",
        "found": "Indica si se encontraron resultados en internet.",
        "context": "Texto construido a partir de los resultados de búsqueda.",
        "sources": "Lista de URLs recuperadas.",
        "reason": "Explicación breve del resultado de la búsqueda."
    }
}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Web Search tool
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def web_search_tool(query: str, max_results: int = 5) -> dict:

    try:
        results = list(
            DDGS().text(
                query=query,
                max_results=max_results
            )
        )

        if not results:
            return {
                "tool_name": "web_search_tool",
                "found": False,
                "context": "",
                "sources": [],
                "reason": "No se encontraron resultados en internet."
            }

        context_parts = []
        sources = []

        for result in results:
            title = result.get("title", "")
            body = result.get("body", "")
            href = result.get("href", "")

            context_parts.append(
                f"Título: {title}\nContenido: {body}\nURL: {href}"
            )

            if href:
                sources.append(href)

        context = "\n\n".join(context_parts)

        return {
            "tool_name": "web_search_tool",
            "found": True,
            "context": context,
            "sources": sources,
            "reason": "Se encontró información en internet usando ddgs."
        }

    except Exception as error:
        return {
            "tool_name": "web_search_tool",
            "found": False,
            "context": "",
            "sources": [],
            "reason": f"Ocurrió un error al buscar en internet: {error}"
        }

In [72]:
web_result = web_search_tool(
    query="What is Retrieval-Augmented Generation?",
    max_results=5
)

print("TOOL:", web_result["tool_name"])
print("FOUND:", web_result["found"])
print("SOURCES:", web_result["sources"])
print("REASON:", web_result["reason"])
print()
print("CONTEXT:")
print(web_result["context"])

TOOL: web_search_tool
FOUND: True
SOURCES: ['https://research.ibm.com/blog/retrieval-augmented-generation-RAG', 'https://blogs.nvidia.com/blog/what-is-retrieval-augmented-generation/', 'https://www.techzine.eu/blogs/applications/116607/what-is-retrieval-augmented-generation/', 'https://www.oracle.com/artificial-intelligence/generative-ai/retrieval-augmented-generation-rag/', 'https://aws.amazon.com/what-is/retrieval-augmented-generation/']
REASON: Se encontró información en internet usando ddgs.

CONTEXT:
Título: What is retrieval-augmented generation (RAG)? - IBM Research
Contenido: Retrieval-augmented generation (RAG) is an AI framework for improving the quality of LLM-generated responses by grounding the model on external ...
URL: https://research.ibm.com/blog/retrieval-augmented-generation-RAG

Título: What Is Retrieval-Augmented Generation aka RAG | NVIDIA Blogs
Contenido: Retrieval-augmented generation is a technique for enhancing the accuracy and reliability of generative AI mod